# Ball detector training

Trains the app's ball detector on Colab's free GPU and exports
`ball_detector.tflite` for `assets/models/`.

**Before running:**
1. *Runtime → Change runtime type → T4 GPU.*
2. In Google Drive, make a folder `BallSpeedAI` with:
   - `prepare_dataset.py` (from the project's `ml/` folder),
   - `videos/` with your bowling videos (phone on a tripod behind the batting stumps, as in the app).
3. Get a free Roboflow API key (roboflow.com → Settings → API Keys) and paste it below. Without one, only your videos are used.

Then *Runtime → Run all*. It takes about 30–60 minutes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/BallSpeedAI'
ROBOFLOW_API_KEY = ''  # paste your key between the quotes
IMGSZ = 320            # the model's input size; the app adapts to whatever is exported
EPOCHS = 120

import os
assert os.path.exists(f'{WORK}/prepare_dataset.py'), 'Copy ml/prepare_dataset.py into BallSpeedAI on Drive'
videos = [f for f in os.listdir(f'{WORK}/videos')] if os.path.isdir(f'{WORK}/videos') else []
print(len(videos), 'videos:', videos[:10])

In [ ]:
!pip -q install ultralytics roboflow opencv-python-headless pyyaml

## Public cricket ball datasets

Downloaded from Roboflow Universe (latest version of each). A dataset that fails to download is skipped.

In [ ]:
PUBLIC = [
    ('learningplatform', 'cricket-ball-detection-rixg7'),
    ('learningplatform', 'cricket-ball-detection-4g7e0'),
    ('drs-test', 'cricket-ball-detection-6csd0'),
]
public_dirs = []
if ROBOFLOW_API_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    for workspace, name in PUBLIC:
        try:
            project = rf.workspace(workspace).project(name)
            latest = max(project.versions(), key=lambda v: int(str(v.version).split('/')[-1]))
            location = f'/content/public/{name}'
            latest.download('yolov8', location=location)
            public_dirs.append(location)
            print('downloaded', name)
        except Exception as error:
            print('skipped', name, '-', error)
else:
    print('No Roboflow key: using your videos only.')

## Build the dataset

Your videos are labelled automatically: the ball is the small patch that moves in a straight line down the pitch.
Check the previews below; if boxes are on something other than the ball, film with less movement around the pitch.

In [ ]:
public_args = ' '.join(f'--public "{d}"' for d in public_dirs)
videos_arg = f'--videos "{WORK}/videos"' if videos else ''
!python "{WORK}/prepare_dataset.py" {public_args} {videos_arg} --out /content/dataset

In [ ]:
import glob
from IPython.display import Image, display
for path in sorted(glob.glob('/content/dataset/preview/*.jpg'))[:12]:
    display(Image(path, width=480))

## Train

YOLO11n, the smallest model, so it runs in real time on a phone. Colour augmentation is off because the app sees only brightness.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
model.train(
    data='/content/dataset/data.yaml',
    imgsz=IMGSZ,
    epochs=EPOCHS,
    patience=30,
    batch=32,
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.4,
    degrees=0.0, fliplr=0.5, scale=0.6, translate=0.1, mosaic=1.0,
    project='/content/runs', name='ball', exist_ok=True,
)

In [ ]:
best = YOLO('/content/runs/ball/weights/best.pt')
metrics = best.val(data='/content/dataset/data.yaml', imgsz=IMGSZ)
print(f'precision {metrics.box.mp:.3f}  recall {metrics.box.mr:.3f}  mAP50 {metrics.box.map50:.3f}')

## Export for the app

Saves `ball_detector.tflite` to `BallSpeedAI` on Drive. Copy it to the project's `assets/models/` and rebuild the app.

In [ ]:
import shutil
# Ultralytics exports TensorFlow Lite as 'litert' (FP32); older versions call it 'tflite'.
try:
    exported = best.export(format='litert', imgsz=IMGSZ)
except Exception:
    exported = best.export(format='tflite', imgsz=IMGSZ)
tflite = sorted(glob.glob('/content/runs/**/*.tflite', recursive=True), key=os.path.getmtime)[-1]
shutil.copy(tflite, f'{WORK}/ball_detector.tflite')
shutil.copy('/content/runs/ball/weights/best.pt', f'{WORK}/ball_detector.pt')
print('Saved', tflite, '->', f'{WORK}/ball_detector.tflite')
